# MeCoBi from Python: build, check and export metal complexes

This notebook is the scripted counterpart of the MeCoBi web GUI. It runs the same
pipeline (`mecobi.pipeline.build_complex`, the code path the GUI and the `mecobi`
CLI both call) and walks through the same steps:

| GUI step | Here |
|---|---|
| Paste ligand SMILES, **Parse** | §2 `inputs.parse_row` + `pipeline.resolve_perception` |
| Donor table / donor correction | §2 donor table, §3.1 `donor_override` |
| Geometry dropdown | §3.2 `Complex.geometry` |
| 3D **Assign** scene (numbered sites) | §3.3 `site_assignment` / isomer notation |
| Enumerate isomers | §3.4 `all_isomers=True`, `stereo` |
| xTB relaxation, energy ranking | §4 `relax=True` |
| **Build all**, batch download | §5 CLI `mecobi build` and the same loop in Python |
| SMILES / notation recovered from the structures | §6 `mecobi.reverse` |
| 3D viewer | after every build, and §7: `py3Dmol` (interactive), `xyzrender` (static images) |

Every cell runs without xTB: the relaxation cells check for the `xtb` binary and skip
themselves when it is absent.

## 1. Install and import

From a clone of the repository:

```bash
pip install -e .                       # MeCoBi core (RDKit, stk, MCHammer, pydentate)
pip install py3Dmol xyzrender          # viewers (optional)
conda install -c conda-forge xtb       # optional: only needed for relax=True
```

For structures identical to the paper and CI, add the lock: `pip install -e . -c constraints.txt`.

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

import mecobi
from mecobi import geometry, inputs, pipeline, reverse, spin
from mecobi.model import Geometry

try:
    import py3Dmol
except ImportError:
    py3Dmol = None
    print("pip install py3Dmol to see the structures in the notebook")

HAVE_XTB = shutil.which("xtb") is not None
OUT = Path("mecobi_notebook_output")  # every structure this notebook writes goes here
OUT.mkdir(exist_ok=True)
FMT = ("xyz", "mol")  # the .mol keeps the bonds of the assembled complex, which the viewer draws
print("MeCoBi", mecobi.__version__, "| xtb found:", HAVE_XTB)

In [ ]:
# Small helpers used throughout: read an .xyz, measure angles/distances, tabulate donors.
def read_xyz(path):
    lines = Path(path).read_text().splitlines()
    n = int(lines[0])
    atoms = [ln.split() for ln in lines[2:2 + n]]
    return [a[0] for a in atoms], np.array([[float(x) for x in a[1:4]] for a in atoms])


def angle(path, i, j, k):
    # Angle i-j-k in degrees (0-based atom indices into the .xyz).
    _, xyz = read_xyz(path)
    u, v = xyz[i] - xyz[j], xyz[k] - xyz[j]
    return np.degrees(np.arccos(u @ v / np.linalg.norm(u) / np.linalg.norm(v)))


def donor_table(cx):
    # The GUI's donor table: which atoms of each ligand SMILES bind the metal.
    from rdkit import Chem

    rows = []
    for i, lig in enumerate(cx.ligands):
        mol = Chem.MolFromSmiles(lig.smiles)
        donors = lig.effective_donors or []
        rows.append({
            "ligand": i, "SMILES": lig.smiles, "donor atoms": donors,
            "elements": [mol.GetAtomWithIdx(a).GetSymbol() for a in donors],
            "denticity": lig.denticity, "eta (pi) sites": lig.pi_sites or "",
            "charge": lig.formal_charge,
        })
    return pd.DataFrame(rows).set_index("ligand")


def ml_distances(r):
    # Metal-donor distances (A) of a build, in donor order.
    _, xyz = read_xyz(r.xyz_path)
    return [round(float(np.linalg.norm(xyz[d] - xyz[r.metal_index])), 2)
            for d in r.donor_indices]


def results_table(results):
    return pd.DataFrame([{
        "isomer": r.isomer_label, "file": Path(r.xyz_path).name,
        "charge": r.total_charge, "uhf": r.uhf,
        "rel. E (kcal/mol)": r.relative_energy_kcal,
        "connectivity ok": r.connectivity_ok,
    } for r in results])


def show(*results, cols=4, size=260):
    # One viewer panel per build. The bonds come from the .mol written beside each .xyz (the bonds
    # of the assembled complex): drawing from the .xyz alone, the viewer guesses bonds from
    # distances and misses or invents metal-ligand bonds (e.g. Fe-Cp, Ru-N, Pt-O).
    if py3Dmol is None:
        return
    cols = min(cols, len(results))
    rows = -(-len(results) // cols)
    v = py3Dmol.view(width=size * cols, height=size * rows, viewergrid=(rows, cols), linked=False)
    for k, r in enumerate(results):
        cell = (k // cols, k % cols)
        mol = next((p for p in r.extra_paths if p.endswith(".mol")), None)
        v.addModel(Path(mol or r.xyz_path).read_text(), "sdf" if mol else "xyz", viewer=cell)  # a .mol is read as "sdf"
        v.setStyle({}, {"stick": {"radius": 0.15}, "sphere": {"scale": 0.25}}, viewer=cell)
        label = Path(r.xyz_path).stem + (f" ({r.isomer_label})" if r.isomer_label else "")
        v.addLabel(label, {"fontSize": 10, "inFront": True, "useScreen": True,
                           "position": {"x": 5, "y": 5}}, viewer=cell)
    v.zoomTo()
    v.show()


## 2. A first complex: $[\mathrm{Pt(py)_4}]^{2+}$ with all defaults

A complex is one CSV row: metal, **oxidation state**, then the free-ligand SMILES
(counter-ions are stripped). The oxidation state matters: it sets the d-electron count,
which picks between square-planar and tetrahedral for CN 4. Pt(II) is d8, so
tetrakis(pyridine)platinum(II) comes out square planar.

In [ ]:
cx = inputs.parse_row(["Pt_py4", "Pt", "2", "c1ccncc1", "c1ccncc1", "c1ccncc1", "c1ccncc1"])
pipeline.resolve_perception(cx)  # pydentate: donor atoms + denticity per ligand
donor_table(cx)

In [ ]:
def proposed_geometry(cx):
    # The geometry MeCoBi proposes from CN, d-electron count and ligand denticities.
    d = spin.d_count_or_none(cx.metal.symbol, cx.metal.oxidation_state)
    geom = geometry.propose_geometry(
        cx.coordination_number, d, denticities=[lig.denticity for lig in cx.ligands],
        lone_pair=spin.has_stereo_active_lone_pair(cx.metal.symbol, cx.metal.oxidation_state),
    )
    options = [g.value for g in geometry.geometries_for_cn(cx.coordination_number)]
    return d, geom, options


d, geom, options = proposed_geometry(cx)
print(f"CN = {cx.coordination_number}, d{d}, total charge = {cx.total_charge}")
print(f"proposed geometry: {geom.value}   (options for this CN: {options})")

In [ ]:
pt_py4 = pipeline.build_complex(cx, OUT, relax=False, formats=FMT)[0]  # one canonical isomer, no xTB
print(Path(pt_py4.xyz_path).name)
print("metal atom:", pt_py4.metal_index, "| donor atoms in the .xyz:", pt_py4.donor_indices)
print("M-donor distances (A):", ml_distances(pt_py4))
show(pt_py4)


## 3. Options

### 3.1 Override the perceived donors (`donor_override`)

Nitrite binds through N (nitro) or through O (nitrito): the linkage isomers
$[\mathrm{Co(NH_3)_5(NO_2)}]^{2+}$ and $[\mathrm{Co(NH_3)_5(ONO)}]^{2+}$. Perception picks
one; `donor_override` pins the donor atoms by their 0-based heavy-atom index in the ligand
SMILES (the GUI's donor correction panel does the same). In `[O-]N=O` atom 0 is O and
atom 1 is N.

In [ ]:
row = ["Co_NH3_5_NO2", "Co", "3", "N", "N", "N", "N", "N", "[O-]N=O"]
nitro = inputs.parse_row(row)
pipeline.resolve_perception(nitro)
print("perceived:")
display(donor_table(nitro).tail(1))

nitrito = inputs.parse_row(["Co_NH3_5_ONO", *row[1:]])
nitrito.ligands[5].donor_override = [0]  # bind nitrite through O instead
pipeline.resolve_perception(nitrito)
print("overridden:")
display(donor_table(nitrito).tail(1))

linkage = []
for c in (nitro, nitrito):
    r = pipeline.build_complex(c, OUT, relax=False, formats=FMT)[0]
    linkage.append(r)
    el, _ = read_xyz(r.xyz_path)
    print(f"{Path(r.xyz_path).name}: nitrite bound through {el[r.donor_indices[5]]};",
          "M-donor distances (A):", ml_distances(r))
show(*linkage)


### 3.2 Choose the geometry

The proposal is a default, not a constraint. For four separate monodentates on a non-d8
centre such as copper(II) (d9), MeCoBi proposes tetrahedral; any other geometry offered for
that coordination number can be requested by setting `Complex.geometry`. Here
$[\mathrm{Cu(NH_3)_4}]^{2+}$ is built both ways, tetrahedral and square planar.


In [ ]:
def max_donor_angle(r):
    # Largest donor-M-donor angle: ~109.5 deg tetrahedral, ~180 deg square planar.
    dn = r.donor_indices
    return max(angle(r.xyz_path, a, r.metal_index, b) for a in dn for b in dn if a < b)


cu = inputs.parse_row(["Cu_NH3_4", "Cu", "2", "N", "N", "N", "N"])
pipeline.resolve_perception(cu)
print("proposed:", proposed_geometry(cu)[1].value)
cu_td = pipeline.build_complex(cu, OUT / "cu_default", relax=False, formats=FMT)[0]

cu.geometry = Geometry.SQUARE_PLANAR
cu_sp = pipeline.build_complex(cu, OUT / "cu_square_planar", relax=False, formats=FMT)[0]
for label, r in (("proposed", cu_td), ("chosen", cu_sp)):
    print(f"{label:10s} largest N-Cu-N = {max_donor_angle(r):5.1f} deg")
show(cu_td, cu_sp)


### 3.3 Build one specific isomer: numbered sites

Every geometry numbers its coordination sites `0 … CN-1` on fixed axes. The GUI's 3D
**Assign** scene and the text isomer notation use the same numbers (figures in
[`docs/sites/`](../docs/sites/), see the README section *Coordination-site numbering*).
Octahedral: `0`=+x `1`=−x `2`=+y `3`=−y `4`=+z `5`=−z, so the trans pairs are (0, 1),
(2, 3) and (4, 5).

In [ ]:
from IPython.display import SVG, display

display(SVG(filename="../docs/sites/octahedral.svg"))

Two equivalent ways to pin the sites, here for Werner's cis/trans pair
$[\mathrm{Co(NH_3)_4(NO_2)_2}]^+$ (ligands 0-3 are NH₃, 4 and 5 are nitro):

* `site_assignment={ligand index: (site, ...)}` passed to `build_complex`: **trans**, nitro
  on sites 4 and 5;
* the isomer notation string on `Complex.isomer_spec` (the CSV `isomer` column), format
  `ligIdx[(donorAtoms)][:sites]` per ligand, `;`-separated: **cis**, nitro on sites 0 and 2.

In [ ]:
row = ["Co_NH3_4_NO2_2", "Co", "3", "N", "N", "N", "N", "[O-]N=O", "[O-]N=O"]
co_trans = pipeline.build_complex(
    inputs.parse_row(["Co_trans", *row[1:]]), OUT, relax=False, formats=FMT,
    site_assignment={0: (0,), 1: (1,), 2: (2,), 3: (3,), 4: (4,), 5: (5,)},
)[0]

cis_cx = inputs.parse_row(["Co_cis", *row[1:]])
cis_cx.isomer_spec = "0:1;1:3;2:4;3:5;4:0;5:2"
co_cis = pipeline.build_complex(cis_cx, OUT, relax=False, formats=FMT)[0]

for r in (co_trans, co_cis):
    m, dn = r.metal_index, r.donor_indices
    print(f"{Path(r.xyz_path).name:20s} O2N-Co-NO2 = {angle(r.xyz_path, dn[4], m, dn[5]):6.1f} deg",
          "| M-donor (A):", ml_distances(r))
show(co_trans, co_cis)


### 3.4 Enumerate every isomer (`all_isomers=True`, `stereo`)

`all_isomers=True` builds every distinct isomer. With `stereo=True` (default) mirror
images are kept apart (Δ/Λ, asymmetric chelate directions); `stereo=False` gives the
coarse cis/trans count. Example: $[\mathrm{Co(en)_2(NO_2)_2}]^+$.

In [ ]:
row = ["Co_en2_NO2_2", "Co", "3", "NCCN", "NCCN", "[O-]N=O", "[O-]N=O"]
isomers = {}
for stereo in (True, False):
    built = isomers[stereo] = pipeline.build_complex(
        inputs.parse_row(row), OUT / f"stereo_{stereo}", relax=False,
        all_isomers=True, stereo=stereo, formats=FMT,
    )
    print(f"stereo={stereo}:", [r.isomer_label for r in built])
show(*isomers[True])  # every isomer, mirror images apart


## 4. Relaxation with xTB (`relax=True`) vs assembly only (`relax=False`)

`relax=False` stops after stk assembly + MCHammer (seconds, no xTB): a starting structure,
not an optimised one. Chelate rings in particular come out with M-donor distances a few
tenths of an ångström long. `relax=True` adds a GFN2-xTB optimisation, always called with
the complex's charge (`--chrg`) and unpaired electrons (`--uhf`). When several isomers are
relaxed, the results come back ordered lowest-energy first, with the energy above the
lowest isomer. An isomer whose bonding changed during relaxation is flagged
`connectivity_ok=False` and not ranked.

These cells skip themselves when `xtb` is not on `PATH`.

In [ ]:
if HAVE_XTB:
    relaxed = pipeline.build_complex(
        inputs.parse_row(["Co_NH3_4_NO2_2", "Co", "3", "N", "N", "N", "N",
                          "[O-]N=O", "[O-]N=O"]),
        OUT / "relaxed", relax=True, all_isomers=True, formats=FMT,
    )
    display(results_table(relaxed))
    show(*relaxed)
else:
    relaxed = None
    print("xtb not found: skipping relaxation (install with conda install -c conda-forge xtb)")

In [ ]:
if relaxed:
    r_cis = next(r for r in relaxed if r.isomer_label == "cis")
    print("Co-donor distances (A), assembled cis:", ml_distances(co_cis))
    print("Co-donor distances (A), relaxed cis:  ", ml_distances(r_cis))

## 5. Batch: a CSV of complexes

Three metallodrugs as a CSV (one complex per row; the `Charge` column is the metal
oxidation state): carboplatin, and ferroquine and TLD1433 from the GUI's
**Load sample** set. An optional double-quoted `isomer` column takes the notation from §3.3.

In [ ]:
SAMPLE = '''Name,Metal,Charge,L1,L2,L3
Carboplatin,Pt,2,N,N,[O-]C(=O)C1(CCC1)C(=O)[O-]
Ferroquine,Fe,2,CN(C)Cc1cc[cH-]c1CNc2ccnc3cc(Cl)ccc23,[cH-]1cccc1,
TLD1433,Ru,2,CC1=CC(=NC=C1)C2=NC=CC(=C2)C,CC1=CC(=NC=C1)C2=NC=CC(=C2)C,C1=CC2=C3C(=C4C=CC=NC4=C2N=C1)N=C(N3)C5=CC=C(S5)C6=CC=C(S6)C7=CC=CS7
'''
csv_path = OUT / "metallodrugs.csv"
_ = csv_path.write_text(SAMPLE)

**CLI.** In a terminal this is `mecobi build metallodrugs.csv --output-dir batch_cli --no-relax`
(drop `--no-relax` to relax with xTB, add `--all-isomers`, `--format xyz,mol,pdb`). Below
it runs through the same entry point. Paths go to stdout, notes and failures to stderr.

In [ ]:
cli = subprocess.run(
    [sys.executable, "-m", "mecobi.cli", "build", str(csv_path),
     "--output-dir", str(OUT / "batch_cli"), "--no-relax"],
    capture_output=True, text=True, check=False,
)
print(cli.stdout)
print("exit status:", cli.returncode)

**Python.** The same batch: `inputs.parse_csv` returns one `Complex` per row; loop over
`build_complex` and keep going past a failure, as the CLI does. `formats` also writes
`.mol`/`.pdb` beside each `.xyz`.

In [ ]:
batch = []
for cx_i in inputs.parse_csv(csv_path):
    try:
        batch += pipeline.build_complex(cx_i, OUT / "batch_py", relax=False, formats=FMT)
    except Exception as exc:  # noqa: BLE001 - report and continue, like the CLI
        print(f"{cx_i.name}: failed: {exc}")
pd.DataFrame([{
    "complex": r.complex.name, "uhf": r.uhf,
    "files": ", ".join(Path(p).name for p in r.extra_paths), "charge": r.total_charge,
} for r in batch])

In [ ]:
show(*batch, cols=3)


## 6. Reverse conversion: structure → SMILES and isomer notation

`mecobi.reverse.reverse_xyz` reads a mononuclear complex `.xyz` and returns the metal,
one SMILES per ligand and its donor atoms; `to_isomer_notation()` gives the donor part of
the notation, ready for an `isomer` column. This is what the GUI's batch download writes
next to the structures. CLI: `mecobi reverse file.xyz --notation`. An η-bound ring lists
all its carbons as donor atoms, so `CN` here counts metal-bonded atoms (12 for
ferroquine, whose two Cp rings take 2 sites), not coordination sites.

In [ ]:
rows = []
for r in batch:
    rc = reverse.reverse_xyz(r.xyz_path)
    rows.append({
        "file": Path(r.xyz_path).name, "metal": rc.metal, "CN": rc.coordination_number,
        "ligand SMILES": ".".join(lig.smiles for lig in rc.ligands),
        "isomer notation": rc.to_isomer_notation(),
    })
pd.DataFrame(rows)

In [ ]:
cli = subprocess.run(
    [sys.executable, "-m", "mecobi.cli", "reverse", co_trans.xyz_path, "--notation"],
    capture_output=True, text=True, check=False,
)
print(cli.stdout)

## 7. Visualisation

### 7.1 Interactive 3D with py3Dmol

In [ ]:
# All the structures above in one grid (bonds from the .mol files, as in every viewer above).
show(pt_py4, *linkage, cu_td, cu_sp, co_trans, co_cis, *batch, cols=4)


### 7.2 Static images with xyzrender

In [ ]:
# xyzrender draws publication-style static SVG/PNG images (pip install xyzrender).
try:
    import xyzrender
except ImportError:
    xyzrender = None
    print("pip install xyzrender for static images")

if xyzrender:
    figures = {"pt_py4": pt_py4.xyz_path, "co_trans": co_trans.xyz_path,
               "ferroquine": batch[2].xyz_path}
    for name, path in figures.items():
        # haptic=True draws a Cp ring as one bond to its centroid instead of five M-C bonds
        img = xyzrender.render(path, haptic=name == "ferroquine", canvas_size=400,
                               output=OUT / f"{name}.svg")
        display(img)  # also saved as mecobi_notebook_output/<name>.svg (.png works too)

---
Structures are written to `mecobi_notebook_output/` (gitignored). The full option list of
`build_complex` is in its docstring (`help(pipeline.build_complex)`); the web GUI exposes
the same switches under *Advanced options*.